In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from data_utils import (
    load_vocab, load_triples_csv, build_edge_list_from_df,
    KGDataset, collate_for_loader, compute_2hop_drug_gene_disease_support
)

from models_new import DistMult, ComplEx, SimpleGraphSAGE, ProposedRGCNModel
from eval_utils_new import compute_classification_metrics, expected_calibration_error



c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\torch_geometric\typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\torch_geometric\typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
 

In [11]:

# Helper: convert df → triples

def df_to_triple_list(df, ent2id, rel2id):
    triples = []
    for _, row in df.iterrows():
        h, rel, t = row['head'], row['relation'], row['tail']
        if h in ent2id and t in ent2id and rel in rel2id:
            triples.append((ent2id[h], rel2id[rel], ent2id[t]))
    return triples


# Training for one epoch (mixed precision)

def train_one_epoch(model, optimizer, train_loader, device, edge_index=None, edge_type=None,
                    pair_support=None, lambda_path=0.0, scaler=None, max_grad_norm=None):
    model.train()
    total_loss = 0.0
    bce = nn.BCEWithLogitsLoss()

    for batch in train_loader:
        heads, rels, tails, labels = [x.to(device) for x in batch]

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            try:
                logits = model(heads, rels, tails)
            except TypeError:
                logits = model((heads, rels, tails), edge_index.to(device), edge_type.to(device))

            if isinstance(logits, tuple):
                logits = logits[0]

            loss_task = bce(logits, labels)

            if pair_support is not None and lambda_path > 0:
                batch_support = torch.tensor([
                    pair_support.get((int(h), int(t)), 0.0) 
                    for h, t in zip(heads.cpu().numpy(), tails.cpu().numpy())
                ], dtype=torch.float32).to(device)
                batch_support = (batch_support > 0).float()
                loss_path_fn = nn.BCEWithLogitsLoss()
                loss_path = loss_path_fn(logits, batch_support)

                loss = loss_task + lambda_path * loss_path
            else:
                loss = loss_task

        if scaler is not None:
            scaler.scale(loss).backward()
            if max_grad_norm:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if max_grad_norm:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        total_loss += loss.item() * heads.size(0)

    return total_loss / len(train_loader.dataset)



In [12]:

# Mini-batch evaluation

def evaluate_in_batches(model, triples, edge_index, edge_type, device, batch_size=256):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for i in range(0, len(triples), batch_size):
            batch = triples[i:i + batch_size]
            heads, rels, tails = zip(*batch)
            heads = torch.tensor(heads).to(device)
            rels = torch.tensor(rels).to(device)
            tails = torch.tensor(tails).to(device)

            try:
                logits = model(heads, rels, tails)
            except TypeError:
                logits = model((heads, rels, tails), edge_index.to(device), edge_type.to(device))

            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend([1] * len(batch))

    return np.array(all_labels), np.array(all_probs)



In [13]:

# Mini-batch test evaluation (MRR, Hits@10)

def evaluate_test_in_batches(model, triples, edge_index, edge_type, device, batch_size=256, k=10):
    model.eval()
    ranks = []

    with torch.no_grad():
        for i in range(0, len(triples), batch_size):
            batch = triples[i:i + batch_size]
            heads, rels, tails = zip(*batch)
            heads = torch.tensor(heads).to(device)
            rels = torch.tensor(rels).to(device)
            tails = torch.tensor(tails).to(device)

            try:
                logits = model(heads, rels, tails)
            except TypeError:
                logits = model((heads, rels, tails), edge_index.to(device), edge_type.to(device))

            # For MRR / Hits@K
            scores = torch.sigmoid(logits).cpu().numpy()
            ranks.extend([1.0 / (np.argmax(scores[i:] >= scores[i]) + 1) for i in range(len(scores))])

    mrr = np.mean(ranks)
    hits_at_k = np.mean([r <= 1.0/k for r in ranks])
    return mrr, hits_at_k



In [14]:

# Save model checkpoint

def save_model(model, name, save_dir="checkpoints", metrics=None):
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, f"{name}.pt")
    torch.save({"model_state": model.state_dict(), "metrics": metrics}, path)
    print(f"[Saved] {name} → {path}")



In [15]:

# Full training + eval loop

def train_and_evaluate(ModelClass, model_name, train_triples, val_triples, test_triples,
                       ent2id, rel2id, edge_index, edge_type, pair_support,
                       device, epochs=5, batch_size=128, lr=1e-3, lambda_path=0.3, dim=64,
                       use_amp=True, max_grad_norm=1.0):

    num_entities, num_relations = len(ent2id), len(rel2id)

    # Dataset & Loader
    train_ds = KGDataset(train_triples)
    collate = lambda b: collate_for_loader(b, num_entities, neg_per_pos=3)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate, num_workers=0, pin_memory=True
    )

    # Model & Optimizer
    model = ModelClass(num_entities, num_relations, dim=dim, dropout=0.3).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None

    print(f"\n=== Training {model_name.upper()} on {device} ===")
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(
            model, optimizer, train_loader, device, edge_index, edge_type,
            pair_support, lambda_path, scaler, max_grad_norm
        )
        print(f"Epoch {epoch}: loss={loss:.4f}")

    
    # Validation
   
    labels, probs = evaluate_in_batches(model, val_triples, edge_index, edge_type, device, batch_size=128)
    auroc, auprc = compute_classification_metrics(labels, probs)
    ece = expected_calibration_error(labels, probs)
    print(f"[Validation] AUROC={auroc:.3f}, AUPRC={auprc:.3f}, ECE={ece:.3f}")

    # Test Evaluation
   
    mrr, hits_at_10 = evaluate_test_in_batches(model, test_triples, edge_index, edge_type, device, batch_size=128)
    print(f"[Test] MRR={mrr:.3f}, Hits@10={hits_at_10:.3f}")

    # Save model
  
    metrics = {"val": {"AUROC": auroc, "AUPRC": auprc, "ECE": ece},
               "test": {"MRR": mrr, "Hits@10": hits_at_10}}
    save_model(model, model_name, save_dir="checkpoints", metrics=metrics)

    return model



In [16]:


if __name__ == "__main__":
    data_dir = r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph"
    data_dir2 = r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ent2id, id2ent, rel2id, id2rel = load_vocab(
        os.path.join(data_dir, "entities.txt"),
        os.path.join(data_dir, "relations.txt")
    )

    train_df = load_triples_csv(os.path.join(data_dir2, "train_inductive.csv"))
    val_df = load_triples_csv(os.path.join(data_dir2, "val_inductive.csv"))
    test_df = load_triples_csv(os.path.join(data_dir2, "test_inductive.csv"))

    train_triples = df_to_triple_list(train_df, ent2id, rel2id)
    val_triples = df_to_triple_list(val_df, ent2id, rel2id)
    test_triples = df_to_triple_list(test_df, ent2id, rel2id)

    edge_index, edge_type = build_edge_list_from_df(train_df, ent2id, rel2id)
    pair_counts, pair_support = compute_2hop_drug_gene_disease_support(train_df, ent2id)

    models_to_train = {
        "distmult": DistMult,
        "complex": ComplEx,
        "graphsage": SimpleGraphSAGE,
        "proposed_rgcn": ProposedRGCNModel
    }

    all_models = {}
    for name, ModelClass in models_to_train.items():
        model = train_and_evaluate(
            ModelClass, name,
            train_triples, val_triples, test_triples,
            ent2id, rel2id,
            edge_index, edge_type,
            pair_support,
            device,
            epochs=10,
            batch_size=128,
            lr=1e-3,
            lambda_path=0.3,
            dim=64,
            use_amp=True,
            max_grad_norm=1.0
        )
        all_models[name] = model

    print("\n=== Training complete ===")


C:\Users\Manasa\AppData\Local\Temp\ipykernel_10296\2289113632.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None



=== Training DISTMULT on cuda ===


C:\Users\Manasa\AppData\Local\Temp\ipykernel_10296\493081792.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


Epoch 1: loss=2.1546
Epoch 2: loss=1.4196
Epoch 3: loss=1.3416
Epoch 4: loss=1.3124
Epoch 5: loss=1.2978
Epoch 6: loss=1.2881
Epoch 7: loss=1.2824
Epoch 8: loss=1.2779
Epoch 9: loss=1.2743
Epoch 10: loss=1.2718


c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


...

[Validation] AUROC=nan, AUPRC=1.000, ECE=0.270
[Test] MRR=1.000, Hits@10=0.000
[Saved] graphsage → checkpoints\graphsage.pt


C:\Users\Manasa\AppData\Local\Temp\ipykernel_10296\2289113632.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None



=== Training PROPOSED_RGCN on cuda ===


C:\Users\Manasa\AppData\Local\Temp\ipykernel_10296\493081792.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
